In [1]:
from utils.JetGraphProducer import JetGraphProducer
from utils.LundTreeUtilities import OnTheFlyNormalizer
import torch


qcd_path = "/pnfs/psi.ch/cms/trivcat/store/user/ceholm/forTraining/QCD_all/"
signal_base_path = "/pnfs/psi.ch/cms/trivcat/store/user/ceholm/forTraining/SVJ_all/"
ttjets_path = "/pnfs/psi.ch/cms/trivcat/store/user/ceholm/forTraining/TTJets_all/"


signals = {
    "SVJ_700_0p3" : signal_base_path + "SVJ_mMed-700GeV_mDark-20GeV_rinv-0.3_alpha-peak/",
    "SVJ_700_0p5" : signal_base_path + "SVJ_mMed-700GeV_mDark-20GeV_rinv-0.5_alpha-peak/",
    "SVJ_700_0p7" : signal_base_path + "SVJ_mMed-700GeV_mDark-20GeV_rinv-0.7_alpha-peak/",

    "SVJ_800_0p3" : signal_base_path + "SVJ_mMed-800GeV_mDark-20GeV_rinv-0.3_alpha-peak/",
    "SVJ_800_0p5" : signal_base_path + "SVJ_mMed-800GeV_mDark-20GeV_rinv-0.5_alpha-peak/",
    "SVJ_800_0p7" : signal_base_path + "SVJ_mMed-800GeV_mDark-20GeV_rinv-0.7_alpha-peak/",

    "SVJ_900_0p3" : signal_base_path + "SVJ_mMed-900GeV_mDark-20GeV_rinv-0.3_alpha-peak/",
    "SVJ_900_0p5" : signal_base_path + "SVJ_mMed-900GeV_mDark-20GeV_rinv-0.5_alpha-peak/",
    "SVJ_900_0p7" : signal_base_path + "SVJ_mMed-900GeV_mDark-20GeV_rinv-0.7_alpha-peak/",

    "SVJ_1000_0p3" : signal_base_path + "SVJ_mMed-1000GeV_mDark-20GeV_rinv-0.3_alpha-peak/",
    "SVJ_1000_0p5" : signal_base_path + "SVJ_mMed-1000GeV_mDark-20GeV_rinv-0.5_alpha-peak/",
    "SVJ_1000_0p7" : signal_base_path + "SVJ_mMed-1000GeV_mDark-20GeV_rinv-0.7_alpha-peak/",

    "SVJ_1500_0p3" : signal_base_path + "SVJ_mMed-1500GeV_mDark-20GeV_rinv-0.3_alpha-peak/",
    "SVJ_1500_0p5" : signal_base_path + "SVJ_mMed-1500GeV_mDark-20GeV_rinv-0.5_alpha-peak/",
    "SVJ_1500_0p7" : signal_base_path + "SVJ_mMed-1500GeV_mDark-20GeV_rinv-0.7_alpha-peak/",

    "SVJ_3000_0p3" : signal_base_path + "SVJ_mMed-3000GeV_mDark-20GeV_rinv-0.3_alpha-peak/",
    "SVJ_3000_0p5" : signal_base_path + "SVJ_mMed-3000GeV_mDark-20GeV_rinv-0.5_alpha-peak/",
    "SVJ_3000_0p7" : signal_base_path + "SVJ_mMed-3000GeV_mDark-20GeV_rinv-0.7_alpha-peak/",
}



# Produce the graphs. The call to JetGraphProducer will produce the graphs and
# save them in the specified directory if it is not already present.
qcd = JetGraphProducer(
        qcd_path, # Path to the directory containing the root files to be processed
        "QCD_large", # Name to be given to the produced dataset (or the name of the dataset if it is already present)
        n_store_jets=2, # Number of jets to process per event, in order of decreasing pT
        # max_events_to_process=100, # Maximum number of events to process (set to None to process all events)
        use_lund_decomp=True, # Whether to use the Lund decomposition
        n_lund_vars=5, # Number of variables to store in the Lund decomposition (3 or 5)
        save_energy_fractions=True, # Whether to save the energy fractions of the pseudojets in the Lund graph
        save_4vectors_in_lund_tree=True, # Whether to save the 4-vectors of the pseudojets in the Lund graph
        weights="xsec", # Weights to be applied to the graphs (None, "xsec")
        verbose=True, # Whether to print the progress of the processing
        label=0, # Label to be assigned to the graphs (for supervised learning)
    )

ttjets = JetGraphProducer(
        ttjets_path, # Path to the directory containing the root files to be processed
        "TTJets_large", # Name to be given to the produced dataset (or the name of the dataset if it is already present)
        n_store_jets=2, # Number of jets to process per event, in order of decreasing pT
        # max_events_to_process=100, # Maximum number of events to process (set to None to process all events)
        use_lund_decomp=True, # Whether to use the Lund decomposition
        n_lund_vars=5, # Number of variables to store in the Lund decomposition (3 or 5)
        save_energy_fractions=True, # Whether to save the energy fractions of the pseudojets in the Lund graph
        save_4vectors_in_lund_tree=True, # Whether to save the 4-vectors of the pseudojets in the Lund graph
        weights="xsec", # Weights to be applied to the graphs (None, "xsec")
        verbose=True, # Whether to print the progress of the processing
        label=0, # Label to be assigned to the graphs (for supervised learning)
    )

signal_graphs = {}
for signal_name, signal_path in signals.items():
    signal_graphs[signal_name] = (JetGraphProducer(
        signal_path,
        signal_name,
        n_store_jets=2,
        # max_events_to_process=100,
        use_lund_decomp=True,
        n_lund_vars=5,
        save_energy_fractions=True,
        save_4vectors_in_lund_tree=True,
        weights="xsec",
        verbose=True,
        label=1,
    ))


In [2]:
# Compute the means and stds for the normalizer (if required)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


means, stds = 0., 0.
weights = []

for graph in qcd:
    means += graph.x.sum(dim=0)*graph.w
    weights.append(graph.w)
    
for graph in ttjets:
    means += graph.x.sum(dim=0)*graph.w
    weights.append(graph.w)

means /= sum(weights)

for graph in qcd:
    stds += ((graph.x - means)**2).sum(dim=0)*graph.w

for graph in ttjets:
    stds += ((graph.x - means)**2).sum(dim=0)*graph.w

stds /= sum(weights)
stds = torch.sqrt(stds)

normalizer = OnTheFlyNormalizer(["x"], means, stds, device=device)

# # import pickle
# # normalizer_dict = {
# #     "means": means,
# #     "stds": stds,
# # }
# # with open("path_to_file.pkl", "wb") as outFile:
# #     pickle.dump(normalizer_dict, outFile)


from data_loading.graphDataLoader import GraphDataLoader

# # Create the data loaders for training and testing. Note how
# # you can concatenate multiple datasets in a single loader and how
# # you can specify the window of each dataset to load.
loader_training = GraphDataLoader(
    [
        [
            qcd_path, # path to the dataset
            "QCD_large", # name of the dataset (you can process the same raw data with multiple settings given each a different name)
        ],
        [
            ttjets_path, # path to the dataset
            "TTJets_large", # name of the dataset (you can process the same raw data with multiple settings given each a different name)
        ],
    ],
    window_to_load=[[0., 0.7], [0., 0.7]],
    batch_size=64,
    normalizer=normalizer,
    device=device,
)

loader_testing = GraphDataLoader(
    [
        [qcd_path, "QCD_large"],
        [ttjets_path, "TTJets_large"],
    ],
    
    window_to_load=[[0.7, 1.], [0.7, 1.]],
    normalizer=normalizer,
    device=device,
)



/work/ceholm/miniforge3/envs/gnn3/lib/python3.11/site-packages/torch_geometric/data/in_memory_dataset.py:300: UserWarning: It is not recommended to directly access the internal storage format `data` of an 'InMemoryDataset'. If you are absolutely certain what you are doing, access the internal storage via `InMemoryDataset._data` instead to suppress this warning. Alternatively, you can access stacked individual attributes of every graph via `dataset.{attr_name}`.
  warnings.warn(msg)
/work/ceholm/miniforge3/envs/gnn3/lib/python3.11/site-packages/torch_geometric/data/in_memory_dataset.py:300: UserWarning: It is not recommended to directly access the internal storage format `data` of an 'InMemoryDataset'. The given 'InMemoryDataset' only references a subset of examples of the full dataset, but 'data' will contain information of the full dataset. The data of the dataset is already cached, so any modifications to `data` will not be reflected when accessing its elements. Clearing the cache no

tensor(False)
tensor(False)


In [ ]:
loaders_signal = {}

for key in signals.keys():
    loaders_signal[key] = GraphDataLoader(
        [
            [signals[key], key],
        ],
        window_to_load=[[0., 1.] for _ in range(1)],
        normalizer=normalizer,
        device=device,
    )

In [ ]:
# loader_signal = GraphDataLoader(
#     [
#         [signals["SVJ_700_0p3"], "SVJ_700_0p3"],
        
#     ],
#     window_to_load=[[0., 1.] for _ in range(1)],
#     normalizer=normalizer,
#     device=device,
# )

alpha = 4

from architectures.GraphAE import graphAE
import time
from tqdm import tqdm

# Create the model
model = graphAE(
    num_features=12,
    architecture=(256, 128, 64, 32, 16, 8, 4),
).to(device)

# Define the optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)


from torch_geometric.utils import batched_negative_sampling, negative_sampling


# Define training function
def train_on_batch(model, batch, optimizer, alpha=torch.tensor(1.).to(device)):
    """
    Train the model for one epoch.

    Args:
    - model (torch.nn.Module): the model to train
    - batch (torch_geometric.data.Data): the batch of data
    - optimizer (torch.optim.Optimizer): the optimizer to use
    - alpha (float): the weight of the features loss in the total loss

    Returns:
    - float: the edges loss
    - float: the features loss
    """

    model.train()
    x = batch.x
    edges = batch.edge_index
    _batch = batch.batch

    edges_loss = model.edges_loss(
        x,
        edges,
        # Negative sampling is important to also show the network
        # examples of non-connected nodes
        batched_negative_sampling(
            edges,
            _batch,
        ),
    )

    features_loss = model.features_loss(x, edges)
    loss = edges_loss + alpha*features_loss

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    return float(edges_loss.detach()), float(features_loss.detach())


# Train the model
losses = []
losses_edges = []
losses_nodes = []
epochs = 1
alpha = torch.tensor(alpha).to(device)

start = time.time()

for epoch in range(epochs):
    for batch in tqdm(loader_training, desc=f"Epoch {epoch}", leave=False):
        #print(batch)
        loss_edges, loss_features = train_on_batch(model, batch, optimizer, alpha)
    
    losses.append((loss_edges+alpha*loss_features))
    losses_edges.append(loss_edges)
    losses_nodes.append((alpha*loss_features))

print(f"--> Training completed in {time.time()-start:.2f} seconds")

In [ ]:
import matplotlib.pyplot as plt

plt.rcParams.update({'figure.figsize': (10, 7)})

plt.plot(losses, label="Total loss", marker="o")
plt.plot(losses_edges, label="Edges loss", marker="o")
plt.plot(losses_nodes, label="Nodes loss", marker="o")

plt.xlabel("Batch number")
plt.ylabel("Loss")
plt.yscale("log")
plt.legend()
plt.show()

model.eval()

losses_testing_qcd_ttjets = []

for graph in tqdm(loader_testing, desc="Testing QCD and TTJets", leave=True):
    x = graph.x.to(device)
    edges = graph.edge_index.to(device)

    edges_loss = model.edges_loss(
        x,
        edges,
        negative_sampling(
            edges,
        ).to(device),
    )

    features_loss = model.features_loss(x, edges)
    loss = edges_loss + alpha*features_loss

    losses_testing_qcd_ttjets.append(float(loss.detach()))
    
losses_testing_signal = []

for graph in tqdm(loader_signal, desc="Testing SVJ", leave=True):
    x = graph.x.to(device)
    edges = graph.edge_index.to(device)

    edges_loss = model.edges_loss(
        x,
        edges,
        negative_sampling(
            edges,
        ).to(device),
    )

    features_loss = model.features_loss(x, edges)
    loss = edges_loss + alpha*features_loss

    losses_testing_signal.append(float(loss.detach()))


plt.hist(losses_testing_qcd_ttjets, bins=50, histtype="step", label="QCD", density=True)
plt.hist(losses_testing_signal, bins=50, histtype="step", label="Signal", density=True)
text_str = "$r_{inv} = 0.3$\n$m_{Med} =$ 700 GeV"
plt.text(0.8, 0.95, text_str, fontsize=10, ha='right', va='top', transform=plt.gca().transAxes,
         bbox=dict(facecolor='white', alpha=0.5, edgecolor='none'))
plt.xlabel("Loss")
plt.ylabel("Density")
plt.legend()
plt.show()


In [ ]:
losses_testing_signal = {}

for signal_name, loader_signal in loaders_signal.items():
    losses_testing_signal[signal_name] = []
    for graph in tqdm(loader_signal, desc=f"Testing SVJ: {signal_name}", leave=True):
        x = graph.x.to(device)
        edges = graph.edge_index.to(device)

        edges_loss = model.edges_loss(
            x,
            edges,
            negative_sampling(
                edges,
            ).to(device),
        )

        features_loss = model.features_loss(x, edges)
        loss = edges_loss + alpha*features_loss

        losses_testing_signal[signal_name].append(float(loss.detach()))


plt.hist(losses_testing_qcd_ttjets, bins=50, histtype="step", label="QCD", density=True)
plt.hist(losses_testing_signal["SVJ_700_0p3"], bins=50, histtype="step", label="Signal", density=True)
text_str = "$r_{inv} = 0.3$\n$m_{Med} =$ 700 GeV"
plt.text(0.8, 0.95, text_str, fontsize=10, ha='right', va='top', transform=plt.gca().transAxes,
         bbox=dict(facecolor='white', alpha=0.5, edgecolor='none'))
plt.xlabel("Loss")
plt.ylabel("Density")
plt.legend()
plt.show()


In [ ]:
# Make labels and scores dictionaries


from sklearn.metrics import roc_auc_score, roc_curve
import copy
import numpy as np

# Create physical weights to use in auc curves 
weights_bkg_auc = []

qcd_for_weights = copy.deepcopy(qcd)                    # deep copies to avoid modifying the graph data
ttjets_for_weights = copy.deepcopy(ttjets)
signals_for_weights = copy.deepcopy(signal_graphs)

# Only create weights for graphs used for tesing
qcd_for_weights = qcd_for_weights[int(0.7*len(qcd_for_weights)):int(1.0*len(qcd_for_weights))]
weights_bkg_auc.append(qcd_for_weights.w.cpu().numpy())

ttjets_for_weights = ttjets_for_weights[int(0.7*len(ttjets_for_weights)):int(1.0*len(ttjets_for_weights))]
weights_bkg_auc.append(ttjets_for_weights.w.cpu().numpy())

weights_bkg_auc_array = np.concatenate(weights_bkg_auc)

# Store the signal weights in a dictionary
weights_sig_auc = {}
all_weights_auc = {}

signals_for_weights = copy.deepcopy(signal_graphs)
for signal_name, signal_data in signals_for_weights.items():
    weights_sig_auc[signal_name] = signal_data.w.cpu().numpy()

    weights_auc_this_sig = [weights_bkg_auc_array, weights_sig_auc[signal_name]]
    all_weights_auc[signal_name] = np.concatenate(weights_auc_this_sig)
    print(len(all_weights_auc[signal_name]))

y_true = {}
y_score = {}
fpr = {}
tpr = {}
auc = {}

for signal_name, signal_losses in losses_testing_signal.items():
    y_true[signal_name] = [0]*len(losses_testing_qcd_ttjets) + [1]*len(signal_losses)
    y_score[signal_name] = losses_testing_qcd_ttjets + signal_losses

    fpr[signal_name], tpr[signal_name], _ = roc_curve(y_true[signal_name], y_score[signal_name], sample_weight=all_weights_auc[signal_name])
    auc[signal_name] = roc_auc_score(y_true[signal_name], y_score[signal_name], sample_weight=all_weights_auc[signal_name])

for signal_name in auc.keys():
    plt.plot(fpr[signal_name], tpr[signal_name], label=f"AUC = {auc[signal_name]:.3f}")
    plt.xlabel("False positive rate")
    plt.ylabel("True positive rate")
    plt.title(f"ROC curve for {signal_name}")
    plt.legend()
    #plt.show()

In [ ]:
import re
import numpy as np
def make_dict(auc_scores):
    dict = {}
    
    for i, key in enumerate(auc_scores.keys()):
        pattern = r"SVJ_(\d+)_0p(\d+)"
        match = re.search(pattern, key)
        if match:
            mMed = int(match.group(1))
            rinv = float(f"0.{match.group(2)}")
            sig_key = (mMed, rinv)
        
        dict[sig_key] = auc_scores[key]
    return dict

auc_dict = make_dict(auc)

file_name = "test"

import plotUtils as plotUtl

plotUtl.plot_table(
        auc_dict,
        "AUC",
        file_name,
        "$m_{Med}$",
        "$r_{inv}$",
    )